# Base Model Evaluation
This notebook trains and evaluates multiple shallow learning models using the reduced SFS feature subset.

The following classifiers are implemented:

- Logistic Regression
- Support Vector Machine (SVM)
- k-Nearest Neighbors (kNN)
- Random Forest

The evaluation metrics are:

- ACC → Accuracy
- SEN → Sensitivity
- SPE → Specificity
- MCC → Matthews Correlation Coefficient
- F1-score

Confusion matrices are also reported for each model.

# Imports

In [57]:
# ============================================================
# IMPORTS
# ============================================================

# Data manipulation
import pandas as pd
import numpy as np

# Feature scaling
from sklearn.preprocessing import StandardScaler

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV


# Metrics
from sklearn.metrics import (

    accuracy_score,
    recall_score,
    f1_score,
    confusion_matrix,
    matthews_corrcoef
)

# Visualization
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

# Load Reduced SFS Dataset
The training stage uses the reduced dataset generated using Sequential Forward Selection (SFS).

This subset contains only the selected features.

In [58]:
# ============================================================
# LOAD REDUCED SFS DATASET
# ============================================================

# Reduced dataset generated previously

X_train = pd.read_csv(
    "sfs_dataset.csv"
)

# Original training dataset
# used only to recover labels

train_df = pd.read_csv(
    "00-train.csv"
)

# Labels

y_train = train_df["Class"]

print("="*60)
print("SFS TRAIN DATASET")
print("="*60)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

SFS TRAIN DATASET
X_train shape: (198, 10)
y_train shape: (198,)


# Load Tuning and External Datasets
The tuning and external datasets must use the EXACT same features selected during SFS.

In [59]:
# ============================================================
# LOAD TUNING AND EXTERNAL DATASETS
# ============================================================

# Load datasets

tuning_df = pd.read_csv(
    "01-tuning.csv"
)

external_df = pd.read_csv(
    "02-external.csv"
)

# Keep SAME SFS variables

X_tuning = tuning_df[
    X_train.columns
]

X_external = external_df[
    X_train.columns
]

# Labels

y_tuning = tuning_df["Class"]

y_external = external_df["Class"]

print("="*60)
print("OTHER DATASETS")
print("="*60)

print("\nX_tuning shape:")
print(X_tuning.shape)

print("\nX_external shape:")
print(X_external.shape)

OTHER DATASETS

X_tuning shape:
(62, 10)

X_external shape:
(11182, 10)


# Feature Scaling
Scaling is required for:

- Logistic Regression
- SVM
- kNN

The scaler is fitted ONLY using the training dataset.

In [60]:
# ============================================================
# FEATURE SCALING
# ============================================================

# Create scaler object

scaler = StandardScaler()

# Learn scaling parameters
# ONLY from training data

scaler.fit(X_train)

# Transform datasets

X_train_scaled = scaler.transform(
    X_train
)

X_tuning_scaled = scaler.transform(
    X_tuning
)

X_external_scaled = scaler.transform(
    X_external
)

print("Scaling completed successfully.")

Scaling completed successfully.


# Evaluation Function
This function computes:

- Accuracy
- Sensitivity
- Specificity
- MCC
- F1-score

using the confusion matrix.

In [61]:
# ============================================================
# EVALUATION FUNCTION
# ============================================================

def evaluate_model(

    y_true,
    y_pred
):

    # Confusion matrix

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    # Extract values

    tn, fp, fn, tp = cm.ravel()

    # Accuracy

    acc = accuracy_score(
        y_true,
        y_pred
    )

    # Sensitivity / Recall

    sen = recall_score(
        y_true,
        y_pred,
        pos_label="APP"
    )

    # Specificity

    spe = tn / (tn + fp)

    # Matthews Correlation Coefficient

    mcc = matthews_corrcoef(
        y_true,
        y_pred
    )

    # Weighted F1-score

    f1 = f1_score(
        y_true,
        y_pred,
        average='weighted'
    )

    # Return all metrics

    return {

        "ACC": acc,
        "SEN": sen,
        "SPE": spe,
        "MCC": mcc,
        "F1": f1,
        "CM": cm
    }

# Logistic Regression

In [62]:
# ============================================================
# LOGISTIC REGRESSION
# ============================================================

# Create model

lr_model = LogisticRegression(
    random_state=42
)

# Train model

lr_model.fit(

    X_train_scaled,
    y_train
)

# Predictions

y_pred_lr = lr_model.predict(
    X_tuning_scaled
)

# Evaluate model

results_lr = evaluate_model(

    y_tuning,
    y_pred_lr
)

print("="*60)
print("LOGISTIC REGRESSION")
print("="*60)

print(results_lr)

LOGISTIC REGRESSION
{'ACC': 0.7580645161290323, 'SEN': 0.7096774193548387, 'SPE': np.float64(0.7096774193548387), 'MCC': 0.5185629788417315, 'F1': 0.757496740547588, 'CM': array([[22,  9],
       [ 6, 25]])}


# Support Vector Machine (SVM)

In [63]:
# ============================================================
# SUPPORT VECTOR MACHINE
# ============================================================

# Create model

svm_model = SVC(
    random_state=42
)

# Train model

svm_model.fit(

    X_train_scaled,
    y_train
)

# Predictions

y_pred_svm = svm_model.predict(
    X_tuning_scaled
)

# Evaluate model

results_svm = evaluate_model(

    y_tuning,
    y_pred_svm
)

print("="*60)
print("SVM")
print("="*60)

print(results_svm)

SVM
{'ACC': 0.7741935483870968, 'SEN': 0.7419354838709677, 'SPE': np.float64(0.7419354838709677), 'MCC': 0.5495319562599505, 'F1': 0.7739583333333333, 'CM': array([[23,  8],
       [ 6, 25]])}


# k-Nearest Neighbors (kNN)

In [64]:
# ============================================================
# K-NEAREST NEIGHBORS
# ============================================================

# Create model

knn_model = KNeighborsClassifier(
    n_neighbors=5
)

# Train model

knn_model.fit(

    X_train_scaled,
    y_train
)

# Predictions

y_pred_knn = knn_model.predict(
    X_tuning_scaled
)

# Evaluate model

results_knn = evaluate_model(

    y_tuning,
    y_pred_knn
)

print("="*60)
print("KNN")
print("="*60)

print(results_knn)

KNN
{'ACC': 0.7741935483870968, 'SEN': 0.8387096774193549, 'SPE': np.float64(0.8387096774193549), 'MCC': 0.5530100413375022, 'F1': 0.7732497387669802, 'CM': array([[26,  5],
       [ 9, 22]])}


# Random Forest

In [65]:
# ============================================================
# RANDOM FOREST
# ============================================================

# Create model

rf_model = RandomForestClassifier(

    n_estimators=100,
    random_state=42
)

# Train model

rf_model.fit(

    X_train,
    y_train
)

# Predictions

y_pred_rf = rf_model.predict(
    X_tuning
)

# Evaluate model

results_rf = evaluate_model(

    y_tuning,
    y_pred_rf
)

print("="*60)
print("RANDOM FOREST")
print("="*60)

print(results_rf)

RANDOM FOREST
{'ACC': 0.7741935483870968, 'SEN': 0.7741935483870968, 'SPE': np.float64(0.7741935483870968), 'MCC': 0.5483870967741935, 'F1': 0.7741935483870968, 'CM': array([[24,  7],
       [ 7, 24]])}


# Model Comparison Table
This table compares all classifiers using the selected evaluation metrics.

In [66]:
# ============================================================
# MODEL COMPARISON
# ============================================================

comparison_df = pd.DataFrame({

    "Model": [

        "Logistic Regression",
        "SVM",
        "kNN",
        "Random Forest"
    ],

    "ACC": [

        results_lr["ACC"],
        results_svm["ACC"],
        results_knn["ACC"],
        results_rf["ACC"]
    ],

    "SEN": [

        results_lr["SEN"],
        results_svm["SEN"],
        results_knn["SEN"],
        results_rf["SEN"]
    ],

    "SPE": [

        results_lr["SPE"],
        results_svm["SPE"],
        results_knn["SPE"],
        results_rf["SPE"]
    ],

    "MCC": [

        results_lr["MCC"],
        results_svm["MCC"],
        results_knn["MCC"],
        results_rf["MCC"]
    ],

    "F1": [

        results_lr["F1"],
        results_svm["F1"],
        results_knn["F1"],
        results_rf["F1"]
    ]
})

print("="*60)
print("MODEL COMPARISON")
print("="*60)

print(comparison_df)

MODEL COMPARISON
                 Model       ACC       SEN       SPE       MCC        F1
0  Logistic Regression  0.758065  0.709677  0.709677  0.518563  0.757497
1                  SVM  0.774194  0.741935  0.741935  0.549532  0.773958
2                  kNN  0.774194  0.838710  0.838710  0.553010  0.773250
3        Random Forest  0.774194  0.774194  0.774194  0.548387  0.774194


# Confusion Matrices

In [67]:
# ============================================================
# CONFUSION MATRICES
# ============================================================

print("="*60)
print("LOGISTIC REGRESSION")
print("="*60)

print(results_lr["CM"])

print("="*60)
print("SVM")
print("="*60)

print(results_svm["CM"])

print("="*60)
print("KNN")
print("="*60)

print(results_knn["CM"])

print("="*60)
print("RANDOM FOREST")
print("="*60)

print(results_rf["CM"])

LOGISTIC REGRESSION
[[22  9]
 [ 6 25]]
SVM
[[23  8]
 [ 6 25]]
KNN
[[26  5]
 [ 9 22]]
RANDOM FOREST
[[24  7]
 [ 7 24]]


# Final External Evaluation
This section evaluates ALL trained classifiers using the external dataset.

The external dataset was not used during:

- training
- feature selection
- model tuning

Therefore, it provides an unbiased estimate of generalization performance.

In [68]:
# ============================================================
# FINAL EXTERNAL EVALUATION
# ============================================================

# ------------------------------------------------------------
# LOGISTIC REGRESSION
# ------------------------------------------------------------

y_external_lr = lr_model.predict(
    X_external_scaled
)

external_lr = evaluate_model(

    y_external,
    y_external_lr
)

# ------------------------------------------------------------
# SVM
# ------------------------------------------------------------

y_external_svm = svm_model.predict(
    X_external_scaled
)

external_svm = evaluate_model(

    y_external,
    y_external_svm
)

# ------------------------------------------------------------
# KNN
# ------------------------------------------------------------

y_external_knn = knn_model.predict(
    X_external_scaled
)

external_knn = evaluate_model(

    y_external,
    y_external_knn
)

# ------------------------------------------------------------
# RANDOM FOREST
# ------------------------------------------------------------

y_external_rf = rf_model.predict(
    X_external
)

external_rf = evaluate_model(

    y_external,
    y_external_rf
)

print("External evaluation completed successfully.")

External evaluation completed successfully.


# External Comparison Table

This table compares the generalization performance of all models using the external dataset.

In [69]:
# ============================================================
# EXTERNAL MODEL COMPARISON
# ============================================================

external_df = pd.DataFrame({

    "Model": [

        "Logistic Regression",
        "SVM",
        "kNN",
        "Random Forest"
    ],

    "ACC": [

        external_lr["ACC"],
        external_svm["ACC"],
        external_knn["ACC"],
        external_rf["ACC"]
    ],

    "SEN": [

        external_lr["SEN"],
        external_svm["SEN"],
        external_knn["SEN"],
        external_rf["SEN"]
    ],

    "SPE": [

        external_lr["SPE"],
        external_svm["SPE"],
        external_knn["SPE"],
        external_rf["SPE"]
    ],

    "MCC": [

        external_lr["MCC"],
        external_svm["MCC"],
        external_knn["MCC"],
        external_rf["MCC"]
    ],

    "F1": [

        external_lr["F1"],
        external_svm["F1"],
        external_knn["F1"],
        external_rf["F1"]
    ]
})

print("="*60)
print("EXTERNAL DATASET RESULTS")
print("="*60)

print(external_df)

EXTERNAL DATASET RESULTS
                 Model       ACC       SEN       SPE       MCC        F1
0  Logistic Regression  0.782508  0.720195  0.720195  0.224582  0.849312
1                  SVM  0.811483  0.688564  0.688564  0.236382  0.867900
2                  kNN  0.698444  0.754258  0.754258  0.181709  0.792159
3        Random Forest  0.834108  0.737226  0.737226  0.279628  0.882519


# External Confusion Matrices

In [70]:
# ============================================================
# EXTERNAL CONFUSION MATRICES
# ============================================================

print("="*60)
print("LOGISTIC REGRESSION")
print("="*60)

print(external_lr["CM"])

print("="*60)
print("SVM")
print("="*60)

print(external_svm["CM"])

print("="*60)
print("KNN")
print("="*60)

print(external_knn["CM"])

print("="*60)
print("RANDOM FOREST")
print("="*60)

print(external_rf["CM"])

LOGISTIC REGRESSION
[[ 296  115]
 [2317 8454]]
SVM
[[ 283  128]
 [1980 8791]]
KNN
[[ 310  101]
 [3271 7500]]
RANDOM FOREST
[[ 303  108]
 [1747 9024]]
